# Semaine 7 — Jour 6 : Human in the loop et décisions traçables

Objectif : Lier une autorisation au brouillon exact.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J6_human_in_loop.ipynb` et `notebooks/week07/teacher/S7_J6_human_in_loop_teacher.ipynb`.

Critère de réussite : Distinguer validation humaine, possession d’un lien et authentification.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Lier une autorisation au brouillon exact.
- Gérer refus, expiration et décision répétée.
- Distinguer validation humaine, possession d’un lien et authentification.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 6 : Human in the loop et décisions traçables


## 1. La validation humaine est un protocole
Une pause dans un workflow ne suffit pas à définir une autorisation. Il faut savoir qui décide, sur quoi, pendant combien de temps et avec quelles conséquences. Le fil rouge exige une proposition stockée, une empreinte de son contenu, une identité de relecture et une date d’expiration. Tous les brouillons sont relus : aucune confiance déclarée par le générateur ne contourne cette étape.

n8n propose plusieurs mécanismes d’attente et de revue humaine. Nous utilisons ici un workflow distinct de réception de décision et un état durable côté backend. Cela évite de conserver une requête HTTP ouverte pendant une heure et laisse la décision arriver dans une nouvelle exécution. Le nœud Wait et la revue d’outils d’agent sont étudiés comme variantes; ils ne sont pas présents dans les exports livrés.

## 2. Ce qui est approuvé doit être stable
Le moteur calcule un SHA-256 du JSON canonique de la proposition. Le relecteur renvoie ce hash avec approve ou reject. Le moteur compare au hash courant. Si le texte a changé, une ancienne décision échoue. Le hash n’est ni une signature du relecteur ni un secret : il lie simplement la décision au contenu attendu.

La date d’expiration est appliquée au moment de décider. Il n’est pas nécessaire d’attendre le passage du scheduler pour refuser une validation tardive. Le scheduler peut ensuite rendre l’état expired visible. Une approbation avant expiration qui est déjà publiée reste une publication valide; une répétition identique renvoie son résultat sans créer une deuxième ligne.

## 3. Identité et rôle dans le lab
Deux tokens distincts séparent les opérations de service des opérations de décision. Le serveur associe le token de revue au seul acteur reviewer-demo. Un champ reviewer envoyé par le client n’est pas accepté comme preuve d’identité. Le workflow n8n de décision utilise un credential de réception protégé et un credential backend réservé à la revue.

Cette identité à token partagé sert à un lab local. Elle n’identifie pas plusieurs personnes et ne fournit pas une gouvernance d’entreprise. Pour un environnement partagé, remplacer ce mécanisme par OIDC, mapping des sujets aux rôles, contrôle par ressource et audit de l’acteur. Une URL de reprise difficile à deviner ne suffit pas à authentifier une personne.

## 4. Refus et répétitions
Reject termine le job sans publication. Répéter reject avec le même acteur et la même proposition est admis. Tenter approve ensuite retourne un conflit; le moteur ne change pas une décision terminale silencieusement. Une correction du brouillon ou une nouvelle validation exige un parcours explicite supplémentaire, hors périmètre du lab.

Le refus ne doit pas être traité comme une erreur transitoire. Il faut afficher la décision au demandeur et permettre une suite métier adaptée. Dans ce projet, le statut suffit; aucun message externe ni aucune notification réelle n’est envoyé.

## 5. Réduction des risques et audit
La publication simulée et la transition published sont commitées ensemble. L’audit contient accepted, draft_created et published ou rejected. Il ne recopie pas le contenu du ticket. Cette table démontre la traçabilité logique; elle n’est pas un journal inviolable puisque le propriétaire de la base peut la modifier.

Le relecteur doit voir le ticket, le brouillon et la conséquence de la décision. Un bouton « oui » sans contexte augmente les validations mécaniques. Les exports n8n ne fournissent pas de portail de revue; le lab utilise GET /jobs et un POST explicite. L’ergonomie d’un portail est une proposition d’amélioration, pas un composant caché.

## À retenir
Une revue sûre lie une identité autorisée, un contenu stable, une fenêtre temporelle et une transition précise. Le clic n’est que l’interface de ce protocole.


## Diagramme du jour

```mermaid
flowchart TD
  A["Brouillon et empreinte"] --> B["Relecteur autorisé"]
  B --> C{"Garde valide ?"}
  C -->|non| D["Conflit ou refus d’accès"]
  C -->|oui| E{"Décision"}
  E -->|approve| F["Publication simulée"]
  E -->|reject| G["État rejected"]
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
import hashlib
import json
proposal = {"category": "general", "draft": "Version A"}
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
approved_hash = digest(proposal)
changed = {**proposal, "draft": "Version B"}
assert digest(changed) != approved_hash
print("Une validation de A n’autorise pas B")


# Lab J6 — Human in the loop et décisions traçables

Depuis la racine du dépôt :

```bash
python3 book/week07/day06/labs/run.py
```

Résultat attendu : **status=published, publication_count=1**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Hash ou signature ?

Expliquer ce que protège proposal_hash et ce qu’il ne protège pas.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Éprouver les gardes

Tester un mauvais hash, une décision expirée et un token de service sur le endpoint de décision.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Traiter le refus

Pourquoi rejected ne doit-il pas rejoindre retry ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Que manque-t-il à un lien de validation secret ?

## Question 2

Comment gérer une double soumission ?


# Challenge

Approuver deux fois la même proposition, puis tenter de la refuser. Vérifier une seule publication et un conflit sur la décision contradictoire.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J6

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.


# Corrigé des exercices

## Exercice 1 — Hash ou signature ?

Il lie la décision au JSON exact du brouillon courant. Il ne prouve pas l’identité, ne chiffre pas le texte et ne rend pas le journal inviolable.

## Exercice 2 — Éprouver les gardes

Mauvais hash :409; date expirée :409; mauvais rôle HTTP :401. Aucun de ces cas ne crée une publication.

## Exercice 3 — Traiter le refus

Le refus est une décision métier valide. Une reprise automatique chercherait à contourner le choix humain. Une nouvelle proposition exige un nouveau parcours explicite.


# Réponses d’entretien

## Question 1

Authentification du relecteur, contrôle de son rôle et du périmètre, liaison au contenu et expiration. La possession d’un lien ne garantit pas l’identité voulue.

## Question 2

Rendre la décision identique idempotente et refuser une décision contradictoire. Stocker décision, acteur et effet dans une transaction locale, puis traiter séparément la livraison externe.


# Solution du challenge

Après process, conserver proposal_hash. Appeler decide deux fois avec approve et reviewer-demo; les deux appels retournent published. Compter publications :1. Appeler reject avec le même hash :Conflict. Le test test_approval_idempotent est la correction exécutable.


# Revue pédagogique

## Acquis à vérifier

- Lier une autorisation au brouillon exact.
- Gérer refus, expiration et décision répétée.
- Distinguer validation humaine, possession d’un lien et authentification.

## Erreurs fréquentes

Confondre réception et achèvement; assimiler un exemple simulé à une évaluation de modèle; déduire une garantie distribuée d’une transaction locale. Pour ce jour, vérifier en particulier : Le refus est une décision métier valide. Une reprise automatique chercherait à contourner le choix humain. Une nouvelle proposition exige un nouveau parcours explicite.

## Barème indicatif sur 10

- 3 points : explication des concepts avec le scénario du support.
- 3 points : reproduction du lab et analyse de sa sortie.
- 2 points : réponses aux exercices, sans confusion sur les garanties.
- 2 points : challenge argumenté et limite opérationnelle identifiée.

Le parcours essentiel peut être validé sans le challenge. Toute publication sans validation bloque la validation du projet intégré, quel que soit le score.


# Correction exécutable du lab

Dans Jupyter, démarrer le kernel depuis la racine du dépôt ou depuis son arbre. Ce fragment recherche la racine contenant le module S7. Il utilise le moteur complet, distinct du petit exemple autonome.



In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/week07/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis le dépôt décompressé")
sys.path.insert(0, str(root))
from examples.week07.demo import scenario
result = scenario(6)
assert result == {'status': 'published', 'publication_count': 1}
print(result)



Les assertions constituent la preuve minimale de ce scénario; les tests de `tests/week07` couvrent les branches complémentaires. Après process, conserver proposal_hash. Appeler decide deux fois avec approve et reviewer-demo; les deux appels retournent published. Compter publications :1. Appeler reject avec le même hash :Conflict. Le test test_approval_idempotent est la correction exécutable.
